# LangGraph와 AgentCore 메모리 - 휴먼 인 더 루프 (단기 메모리)

## 소개
이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LangGraph와 통합하여 **휴먼 인 더 루프** 워크플로우를 만드는 방법을 시연합니다. 사람의 개입을 위해 에이전트 실행을 중단하는 기능과 결합된 **단기 메모리** 지속성에 초점을 맞추어, 원활한 핸드오프가 가능한 정교한 고객 지원 시나리오를 만듭니다.

## 튜토리얼 세부 정보

| 정보                | 세부 사항                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 사용 사례  | 휴먼 에스컬레이션이 포함된 고객 지원                                             |
| 에이전트 프레임워크 | Langgraph                                                                        |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Langgraph 체크포인터, 휴먼 인 더 루프                    |
| 예제 복잡도         | 초급                                                                             |

학습 내용:
- 워크플로우 지속성을 위해 AgentCore 메모리로 메모리 체크포인터 생성
- 휴먼 인 더 루프 워크플로우를 위한 LangGraph의 인터럽트 메커니즘 사용
- 사람의 개입을 위해 실행을 일시 중지할 수 있는 도구 구현
- LangGraph Command를 사용하여 사람의 입력 후 에이전트 워크플로우 재개
- 원활한 핸드오프가 가능한 복잡한 고객 지원 시나리오 관리

### 시나리오 맥락

이 예제에서는 복잡한 문제를 사람 관리자에게 에스컬레이션할 수 있는 "**고객 지원 에이전트**"를 만듭니다. 에이전트가 사람의 전문 지식이 필요한 상황을 만나면 실행을 일시 중지하고, 현재 상태를 AgentCore 메모리에 저장하고, 사람의 개입을 기다립니다. 그런 다음 사람 관리자가 안내를 제공하면 에이전트가 향상된 맥락으로 재개됩니다.

## 아키텍처
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

### 통합 작동 방식

휴먼 인 더 루프 워크플로우를 위한 LangGraph와 AgentCore 메모리 간의 통합은 다음을 포함합니다:

1. 영구 상태 관리를 위한 체크포인터 백엔드로 AgentCore 메모리 사용
2. 특정 지점에서 실행을 일시 중지하는 인터럽트 메커니즘 구현
3. 추가 맥락과 함께 워크플로우를 재개할 수 있는 사람 관리자 지원
4. 중단 간에 대화 이력 및 상태 유지

이 접근 방식은 AI 에이전트와 사람 관리자가 원활하게 협력하는 지원 워크플로우를 만듭니다.

환경 설정을 시작합니다!

In [ ]:
# 필요한 라이브러리 설치
!pip install -qr requirements.txt

In [ ]:
# LangGraph 및 LangChain 컴포넌트 임포트
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# 휴먼 인 더 루프 구현을 위한 임포트
from langgraph.types import Command, interrupt

In [ ]:
import os
import logging

from bedrock_agentcore.memory import MemoryClient
# 체크포인터로 사용할 AgentCoreMemorySaver 임포트
from langgraph_checkpoint_aws import AgentCoreMemorySaver

logging.getLogger("support-agent").setLevel(logging.INFO)
region = os.getenv('AWS_REGION', 'us-west-2')

logger = logging.getLogger("support-agent")

## 1단계: 메모리 생성
이 섹션에서는 AgentCore 메모리 SDK를 사용하여 메모리 저장소를 생성합니다. 이 메모리는 LangGraph 체크포인터의 백엔드 역할을 하며 영구적인 휴먼 인 더 루프 워크플로우를 가능하게 합니다.

In [ ]:
memory_name = "SupportAgent"

client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory["id"]

### AgentCore 메모리 구성

이제 AgentCore 메모리 체크포인터를 구성하고 LLM을 초기화합니다:

- `memory_id`는 체크포인트가 저장될 AgentCore 메모리 리소스에 해당합니다
- `region`은 리소스의 AWS 리전을 지정합니다
- `MODEL_ID`는 LangGraph 에이전트를 구동할 Bedrock 모델을 정의합니다

`memory_id`와 추가 boto3 클라이언트 키워드 인수(이 경우 `region`)를 사용하여 체크포인터를 인스턴스화합니다.

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# 상태 지속성을 위한 체크포인터 초기화
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

## 2단계: 휴먼 인 더 루프 도구
지원 에이전트가 사용할 도구를 정의합니다. LangGraph의 `interrupt` 타입을 사용하면 에이전트 그래프 실행을 중단하여 사람이 개입하고 쿼리에 응답하여 실행을 계속할 수 있습니다.

In [ ]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b

@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply, human_assistance]

## 3단계: LangGraph 에이전트 구현

이제 AgentCore 메모리 체크포인터와 휴먼 인 더 루프 기능을 갖춘 LangGraph의 `create_react_agent` 빌더로 지원 에이전트를 생성합니다:

In [ ]:
# LLM 초기화
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## 4단계: 지원 에이전트 실행
이제 AgentCore 메모리 체크포인터와 휴먼 인 더 루프 통합으로 에이전트를 실행할 수 있습니다. 이 예제에서는 명시적으로 사용자 지원을 요청합니다. 실제로는 여러 조건에 의해 트리거될 수 있으며, 예를 들어 안전 플래그가 특정 키워드가 사용되면 대화를 사람에게 라우팅할 수 있습니다.

### 구성 설정
LangGraph에서 config는 호출 시 필요한 속성(예: 사용자 ID 또는 세션 ID)을 포함하는 `RuntimeConfig`입니다. [추가 정보는 여기에서 확인](https://langchain-ai.github.io/langgraphjs/how-tos/configuration/)할 수 있습니다.

AgentCore 메모리 체크포인터(`AgentCoreMemorySaver`)의 경우 다음을 지정해야 합니다:
- `thread_id`: AgentCore session_id에 매핑됩니다 (고유한 대화 스레드)
- `actor_id`: AgentCore actor_id에 매핑됩니다 (사용자, 에이전트 또는 기타 식별자)

### 그래프 호출 입력
`inputs` 인수로 최신 사용자 메시지만 전달하면 됩니다. 다른 상태 변수도 포함할 수 있지만 간단한 `create_react_agent`에서는 메시지만 필요합니다.

In [ ]:
# "고객 서비스 담당자와 상담하고 싶습니다."라는 의미의 입력
user_input = "I would like to work with a customer service human agent."
config = {"configurable": {"thread_id": "1", "actor_id": "demo-notebook"}}

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

### 워크플로우 중단

사람 지원 도구가 호출되었을 때 실행이 일시 중지된 것에 주목하세요. 워크플로우가 어디서 중지되었는지 현재 상태를 검사합니다:

In [ ]:
snapshot = graph.get_state(config)
snapshot.next

### 사람 관리자 개입

이제 사람 관리자 역할로서 LangGraph `Command`를 사용하여 응답을 보내 워크플로우를 재개합니다. AgentCore 메모리 체크포인터가 전체 대화 상태를 보존했으므로 채팅을 재개할 수 있습니다.

In [ ]:
human_response = (
    "I'm sorry to hear that you are frustrated. Looking at the past conversation history, I can see that you've requested a refund. I've gone ahead and credited it to your account."
)

human_command = Command(resume={"messages": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

## 요약

이 노트북에서 시연한 내용:

1. 휴먼 인 더 루프 워크플로우를 위한 AgentCore 메모리 리소스 생성 방법
2. 인터럽트 기능을 갖춘 LangGraph 에이전트 구축
3. 사람의 개입을 위해 실행을 일시 중지할 수 있는 도구 구현
4. 중단 중 워크플로우 상태를 유지하기 위한 AgentCoreMemorySaver 사용
5. 사람이 제공한 맥락으로 에이전트 실행 재개

이 통합은 LangGraph의 휴먼 인 더 루프 기능과 AgentCore 메모리의 강력한 상태 지속성을 결합하여 AI 에이전트와 사람 관리자가 원활하게 협력하는 정교한 고객 지원 워크플로우를 만드는 강력함을 보여줍니다.

시연된 접근 방식은 다단계 에스컬레이션, 전문 인력 라우팅, 복잡한 승인 워크플로우를 포함한 더 복잡한 시나리오로 확장할 수 있습니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제합니다.

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)